# Character-Level Text Generation with TensorFlow

This notebook trains a **GRU-based recurrent neural network** to generate Shakespeare-style text, one character at a time.

### Pipeline

1. **Data Loading** — Read the full Shakespeare corpus (~1.1M chars) and build a character-level vocabulary (~65 unique chars)
2. **Sequence Preparation** — Encode text as integers, create sliding windows of 120 chars, and split each into input/target pairs (shifted by 1 position)
3. **Model Architecture** — `Embedding(64)` → `GRU(1024, stateful)` → `Dense(vocab_size)` outputting raw logits
4. **Training** — Optimize with Adam + sparse categorical crossentropy for 30 epochs
5. **Text Generation** — Rebuild model with `batch_size=1`, then autoregressively sample characters using temperature-controlled softmax

### Key Concepts

* **Stateful GRU** — hidden state persists across batches, letting the model learn long-range dependencies
* **Temperature sampling** — controls randomness: `temp < 1` = conservative/repetitive, `temp > 1` = creative/chaotic
* **No GPU required** — model is small enough for CPU training (~30-60 min on Standard_D8ds_v4), but GPU cuts this to ~3-5 min

In [0]:
import numpy as np
import pandas as pd
%matplotlib inline
import matplotlib.pyplot as plt
import tensorflow as tf

In [0]:
import os

# Read shakespeare.txt from the same folder as this notebook
_user = spark.sql("SELECT current_user()").first()[0]
file_path = f"/Workspace/Users/{_user}/databricks_ai/DeepLearning/shakespeare.txt"

with open(file_path, 'r') as f:
    text = f.read()

# Build character-level vocabulary and encoding mappings
vocab = sorted(set(text))
print(f'{len(vocab)} unique characters')
char_to_ix = { ch:i for i,ch in enumerate(vocab) }  # char -> integer index
ix_to_char = np.array(vocab)                         # integer index -> char
encoded_text = np.array([char_to_ix[c] for c in text])  # encode full text as int array

# Create training sequences: each is seq_len+1 chars (input + 1-char-shifted target)
seq_len = 120
total_num_seq = len(text)//(seq_len+1)

char_dataset = tf.data.Dataset.from_tensor_slices(encoded_text)
sequences = char_dataset.batch(seq_len+1, drop_remainder=True)

def split_input_target(chunk):
    """Split a sequence into input (first N chars) and target (shifted by 1)."""
    input_text = chunk[:-1]
    target_text = chunk[1:]
    return input_text, target_text

dataset = sequences.map(split_input_target)

# Shuffle and batch for training
BATCH_SIZE = 128
BUFFER_SIZE = 10000
dataset = dataset.shuffle(BUFFER_SIZE).batch(BATCH_SIZE, drop_remainder=True)

In [0]:
from tensorflow.keras.losses import sparse_categorical_crossentropy

vocab_size = len(vocab)
embedding_dim = 64   # dimension of character embedding vectors
rnn_units = 1024     # GRU hidden state size

def sparse_cat_loss(y_true, y_pred):
    """Wrapper for sparse crossentropy with from_logits=True (model outputs raw logits)."""
    return sparse_categorical_crossentropy(y_true, y_pred, from_logits=True)

def build_model(vocab_size, embedding_dim, rnn_units, batch_size):
    """Build a character-level text generation model: Embedding -> GRU -> Dense."""
    model = tf.keras.Sequential([
        # Map each character index to a dense embedding vector
        tf.keras.layers.Embedding(vocab_size, embedding_dim,
                                  batch_input_shape=[batch_size, None]),
        # Recurrent layer: stateful GRU maintains hidden state across batches
        tf.keras.layers.GRU(rnn_units,
                            return_sequences=True,
                            stateful=True,
                            recurrent_initializer='glorot_uniform'),
        # Output layer: logits over the full character vocabulary at each time step
        tf.keras.layers.Dense(vocab_size)
    ])
    model.compile(optimizer='adam', loss=sparse_cat_loss)
    return model

model = build_model(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    rnn_units=rnn_units,
    batch_size=BATCH_SIZE)
model.summary()

In [0]:
# Train the character-level GRU on Shakespeare text
# 30 epochs is sufficient for coherent text generation on CPU (~30-60 min)
epochs = 30

model.fit(dataset, epochs=epochs)

In [0]:
def generate_text(model, start_seed, gen_size=500, temp=1.0):
    """
    Generate text character-by-character from a trained model.
    
    Args:
        model: Trained Keras model (must be rebuilt with batch_size=1)
        start_seed: Initial string to seed the generation
        gen_size: Number of characters to generate
        temp: Sampling temperature (lower = more deterministic, higher = more creative)
    """
    num_generate = gen_size
    input_eval = [char_to_ix[s] for s in start_seed]
    input_eval = tf.expand_dims(input_eval, 0)

    text_generated = []
    temperature = temp

    # Reset GRU hidden state before generating
    model.reset_states()

    for i in range(num_generate):
        # Get logits for the next character
        predictions = model(input_eval)
        predictions = tf.squeeze(predictions, 0)  # remove batch dim

        # Scale logits by temperature and sample from categorical distribution
        predictions = predictions / temperature
        predicted_id = tf.random.categorical(predictions, num_samples=1)[-1, 0].numpy()

        # Feed predicted character back as next input
        input_eval = tf.expand_dims([predicted_id], 0)
        text_generated.append(ix_to_char[predicted_id])

    return (start_seed + ''.join(text_generated))

# Generate 1000 characters of Shakespeare-style text starting with "ROMEO: "
print(generate_text(model, start_seed="ROMEO: ", gen_size=1000))